# 03B · Preparación de datos de siniestros con víctimas
### Escenario metodológico comparable antes y después de la Ley 2251 de 2022

> **Evidencia ejecutada canónica.** Se eligió conservar este notebook oficial
> sin salidas para no reentrenar ni reescribir el cierre. Su copia ejecutada está
> en [`reports/reproducibilidad/flujo_completo_20261001_02/notebooks/03B_Preparacion_Datos_Con_Victimas.ipynb`](../reports/reproducibilidad/flujo_completo_20261001_02/notebooks/03B_Preparacion_Datos_Con_Victimas.ipynb).
> `resultado.json` registra 9/9 celdas, cero errores y coincidencia exacta del dataset.

Este notebook reconstruye el dataset analítico utilizando exclusivamente siniestros clasificados como `Con Heridos` o `Con Muertos`. Se conserva por separado el escenario original para garantizar trazabilidad.

## Justificación

El artículo 16 de la Ley 2251 de 2022 modificó el artículo 143 de la Ley 769 de 2002. Desde julio de 2022, los accidentes con solamente daños materiales dejaron de requerir la elaboración del IPAT por parte de las autoridades. La Circular Externa 20224000000057 del Ministerio de Transporte, del 29 de septiembre de 2022, impartió instrucciones para su aplicación.

En la base, los registros `Solo Daños` disminuyen de 12.557 en 2022 a 1.115 en 2023, mientras los siniestros con heridos o muertos permanecen relativamente estables. Para evitar mezclar dos regímenes de observación, este escenario restringe el universo a eventos con víctimas.

Fuentes oficiales:

- Ley 2251 de 2022: https://www.secretariasenado.gov.co/senado/basedoc/ley_2251_2022.html
- Secretaría Distrital de Movilidad: https://www.movilidadbogota.gov.co/preguntas-frecuentes/donde-se-consultan-los-reportes-oficiales-de-accidentes-de-transito-en-los-que

## 1. Importación y rutas

In [ ]:
from pathlib import Path
import os, sys

import holidays
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
ROOT = Path(os.environ['SINIESTRALIDAD_ROOT']).expanduser().resolve() if 'SINIESTRALIDAD_ROOT' in os.environ else next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/preparacion.py').exists()), None)
if ROOT is None: raise RuntimeError('Abra desde la raíz/notebooks o defina SINIESTRALIDAD_ROOT.')
sys.path.insert(0, str(ROOT))
from src.preparacion import filtrar_victimas, assign_time_slot, crear_cuadricula, agregar_historicos, etiquetar_dataset
RAW_FILE = ROOT / 'data/raw/base-anuario-de-siniestralidad-2024.xlsx'
PROCESSED_PATH = ROOT / 'data/processed'
REPORTS_PATH = ROOT / 'reports/data_quality'
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
REPORTS_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = PROCESSED_PATH / 'dataset_victimas_localidad_franja_fecha.parquet'

## 2. Carga y diagnóstico del cambio de cobertura

In [ ]:
usecols = [
    'Codigo_Accidente', 'Fecha_Acc', 'AA_Acc', 'MM_Acc', 'Hora_Acc',
    'Localidad', 'Gravedad_Indicador_Tradicional'
]
siniestros = pd.read_excel(RAW_FILE, sheet_name='Siniestros', usecols=usecols)
siniestros['Fecha_Acc'] = pd.to_datetime(siniestros['Fecha_Acc'])
assert siniestros['Codigo_Accidente'].is_unique
print(f'Base original: {len(siniestros):,} registros')
print('Categorías de gravedad:')
print(siniestros['Gravedad_Indicador_Tradicional'].value_counts(dropna=False))

In [ ]:
coverage_audit = (
    siniestros[siniestros['AA_Acc'].between(2018, 2024)]
    .groupby(['AA_Acc', 'Gravedad_Indicador_Tradicional'])
    .size().unstack(fill_value=0)
    .rename_axis(index='Anio')
)
coverage_audit['Total'] = coverage_audit.sum(axis=1)
coverage_audit['Con_Victimas'] = coverage_audit.get('Con Heridos', 0) + coverage_audit.get('Con Muertos', 0)
coverage_audit['Variacion_Solo_Danos'] = coverage_audit['Solo Daños'].pct_change()
coverage_audit['Variacion_Con_Victimas'] = coverage_audit['Con_Victimas'].pct_change()
coverage_audit.to_csv(REPORTS_PATH / 'comparacion_cobertura_por_gravedad.csv', encoding='utf-8-sig')
coverage_audit

La estabilidad de `Con Heridos` y `Con Muertos` frente a la caída de `Solo Daños` respalda la construcción de un universo comparable. Esto no implica que todos los siniestros con víctimas sean observados, sino que su mecanismo de registro no fue afectado de la misma manera por la exclusión legal del IPAT para eventos de solo daños.

## 3. Selección del universo con víctimas

In [ ]:
sin = filtrar_victimas(siniestros)
assert sin[['Localidad', 'Hora_Acc', 'Fecha_Acc']].isna().sum().sum() == 0
print(f'Siniestros con víctimas 2018–2024: {len(sin):,}')
display(sin.groupby(['AA_Acc', 'Gravedad_Indicador_Tradicional']).size().unstack(fill_value=0))

## 4. Franja horaria y conteo diario

In [ ]:
sin['Franja_Horaria'] = sin['Hora_Acc'].map(assign_time_slot)
sin['Fecha_Acc'] = sin['Fecha_Acc'].dt.normalize()
event_count = (sin.groupby(['Localidad', 'Franja_Horaria', 'Fecha_Acc'])
               .size().rename('Num_Accidentes').reset_index())
print(f'Combinaciones con al menos un siniestro con víctimas: {len(event_count):,}')

## 5. Grid completo y variables de calendario

In [ ]:
localities = sorted(sin['Localidad'].unique())
time_slots = ['Madrugada', 'Mañana', 'Tarde', 'Noche']
dates = pd.date_range('2018-01-01', '2024-12-31', freq='D')
dataset = crear_cuadricula(event_count, localities)
print(f'Grid: {len(dataset):,} filas; localidades={len(localities)}; fechas={len(dates)}')

## 6. Variables históricas sin fuga de información

In [ ]:
dataset = agregar_historicos(dataset)

## 7. Etiqueta Alto Riesgo

El umbral es el percentil 66 del número diario de siniestros con víctimas, calculado por localidad y franja exclusivamente con 2018–2022. El mismo umbral congelado se aplica a 2023–2024. En este escenario, `Alto_Riesgo` significa frecuencia elevada de siniestros con personas heridas o fallecidas respecto al historial de esa localidad y franja.

In [ ]:
TRAIN_END = pd.Timestamp('2022-12-31')
dataset, risk_thresholds = etiquetar_dataset(dataset, TRAIN_END)
print('Distribución de umbrales:')
display(risk_thresholds['Umbral_P66_Train'].value_counts().sort_index())
display(dataset.groupby('Periodo')['Alto_Riesgo'].agg(['count', 'sum', 'mean']))

### Control de sensibilidad de la etiqueta

Como los conteos son discretos, se informa cuántas combinaciones reciben umbral 0, 1 o 2. Un umbral 0 hace que `Alto_Riesgo=1` equivalga a la ocurrencia de al menos un siniestro con víctimas; un umbral 1 exige dos o más. Esta heterogeneidad forma parte de la definición relativa por localidad y franja y debe explicarse al interpretar el modelo.

## 8. Validación y exportación

In [ ]:
dataset = dataset.drop(columns='Umbral_P66_Train')
final_cols = [
    'Localidad', 'Franja_Horaria', 'Fecha_Acc', 'Num_Accidentes', 'Dia_Semana', 'Mes',
    'Es_Fin_de_Semana', 'Es_Festivo', 'Accidentes_Prom_7d', 'Accidentes_Prom_30d',
    'Accidentes_Semana_Anterior', 'Sin_Historial_Accidentes_Prom_7d',
    'Sin_Historial_Accidentes_Prom_30d', 'Sin_Historial_Accidentes_Semana_Anterior',
    'Periodo', 'Alto_Riesgo'
]
dataset = dataset[final_cols]
key = ['Localidad', 'Franja_Horaria', 'Fecha_Acc']
assert dataset.shape == (204560, 16)
assert dataset.duplicated(key).sum() == 0
assert dataset.isna().sum().sum() == 0
assert dataset['Localidad'].nunique() == 20
assert dataset['Franja_Horaria'].nunique() == 4
assert dataset['Fecha_Acc'].nunique() == 2557
dataset['Fecha_Acc'] = dataset['Fecha_Acc'].astype(str)
dataset.to_parquet(OUTPUT_FILE, index=False)
print('Guardado:', OUTPUT_FILE.resolve())
print('Dimensiones:', dataset.shape)
print('Nulos:', int(dataset.isna().sum().sum()), '| Duplicados de clave:', int(dataset.duplicated(key).sum()))

## Resultado

El archivo exportado conserva la misma interfaz de 16 columnas del dataset original, pero `Num_Accidentes`, las variables históricas y `Alto_Riesgo` se refieren exclusivamente a siniestros con heridos o muertos. Esto permite adaptar los notebooks de modelado y evaluación cambiando la ruta de entrada, sin mezclar los modelos de ambos escenarios.